In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

file_path = "/content/train_ver2.csv"
if not os.path.exists(file_path):
    raise FileNotFoundError(
        "Upload train_ver2.csv to Colab and make sure it is saved at /content/train_ver2.csv."
    )

header = pd.read_csv(file_path, nrows=0).columns.tolist()
product_cols = [
    c for c in header
    if c.startswith("ind_") and c.endswith("_ult1")
]

required_cols = [
    "fecha_dato", "ncodpers", "nomprov", "renta", "sexo", "age",
    "segmento", "ind_empleado", "tiprel_1mes"
]
usecols = [c for c in required_cols if c in header] + product_cols

if "fecha_dato" not in usecols or "ncodpers" not in usecols:
    raise ValueError("The dataset is missing required columns.")

parts = []
for chunk in pd.read_csv(
    file_path,
    usecols=usecols,
    chunksize=100_000,
    low_memory=False
):
    chunk = chunk[chunk["fecha_dato"].isin(["2016-04-28", "2016-05-28"])]
    if not chunk.empty:
        parts.append(chunk)

data = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

previous_month = data[data["fecha_dato"] == "2016-04-28"].copy()
last_month = data[data["fecha_dato"] == "2016-05-28"].copy()
print("Previous-month rows:", len(previous_month))
print("Last-month rows:", len(last_month))
print("Product columns:", len(product_cols))


Previous-month rows: 928274
Last-month rows: 931453
Product columns: 24


## Q1. City statistics for May 28, 2016

In [ ]:
q1_cols = ["ncodpers", "nomprov", "renta", "age"]
missing = [c for c in q1_cols if c not in last_month.columns]
if missing:
    raise ValueError(f"Q1 requires columns not found in the dataset: {missing}")

q1 = last_month[q1_cols].copy()
q1["renta"] = pd.to_numeric(q1["renta"], errors="coerce")
q1["age"] = pd.to_numeric(q1["age"], errors="coerce")

city_stats = (
    q1.dropna(subset=["nomprov"])
    .groupby("nomprov")
    .agg(
        customer_count=("ncodpers", "nunique"),
        average_income=("renta", "mean"),
        median_income=("renta", "median"),
        average_age=("age", "mean")
    )
    .reset_index()
)

city_stats = city_stats[city_stats["customer_count"] >= 50]
top5_cities = city_stats.sort_values(
    "average_income", ascending=False
).head(5)

print("Q1: Top 5 cities by average income")
display(top5_cities.round(2))
print("Cities with at least 50 customers:", len(city_stats))


Q1: Top 5 cities by average income


,nomprov,customer_count,average_income,median_income,average_age
15,CEUTA,513,200515.89,128699.58,44.01
30,MADRID,298841,178362.99,138024.14,45.21
7,"BALEARS, ILLES",9143,171890.95,122837.22,39.75
8,BARCELONA,88754,164584.84,130877.16,39.87
32,MELILLA,641,149071.94,117718.36,45.33


Cities with at least 50 customers: 52


## Q2. Top 5 products with the most new subscriptions

In [ ]:
if not product_cols:
    raise ValueError("No product indicator columns were found.")

prev = previous_month.drop_duplicates("ncodpers").set_index("ncodpers")
curr = last_month.drop_duplicates("ncodpers").set_index("ncodpers")

common_ids = prev.index.intersection(curr.index)
prev_products = prev.loc[common_ids, product_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)
curr_products = curr.loc[common_ids, product_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

new_products = (curr_products == 1) & (prev_products == 0)
new_counts = new_products.sum().sort_values(ascending=False)

top5_products = new_counts.head(5).rename_axis("product").reset_index(
    name="new_subscriptions"
)

print("Q2: Top 5 products with the most new subscriptions")
display(top5_products)
print("Customers compared:", len(common_ids))


Q2: Top 5 products with the most new subscriptions


,product,new_subscriptions
0,ind_recibo_ult1,10163
1,ind_nom_pens_ult1,5513
2,ind_nomina_ult1,5488
3,ind_tjcr_fin_ult1,4248
4,ind_cco_fin_ult1,3854


Customers compared: 926663


## Q3. Chi-square tests for having at least 3 products

In [ ]:
q3 = last_month.copy()
q3["product_count"] = q3[product_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0).sum(axis=1)
q3["target"] = np.where(
    q3["product_count"] >= 3,
    "3_or_more",
    "fewer_than_3"
)

q3["age_group"] = pd.cut(
    pd.to_numeric(q3["age"], errors="coerce"),
    bins=[0, 30, 65, float("inf")],
    labels=["young (<30)", "adult (30-64)", "senior (65+)"],
    right=False
)

features = {
    "Gender": "sexo",
    "Age group": "age_group",
    "Segment": "segmento",
    "Employment status": "ind_empleado",
    "Customer relation type": "tiprel_1mes"
}

results = []
for name, column in features.items():
    if column not in q3.columns:
        continue

    temp = q3[[column, "target"]].dropna()
    table = pd.crosstab(temp[column], temp["target"])

    if table.shape[0] < 2 or table.shape[1] < 2:
        continue

    chi2, p_value, dof, expected = chi2_contingency(table)
    results.append({
        "Feature": name,
        "Chi-square": chi2,
        "Degrees of freedom": dof,
        "P-value": p_value,
        "Significant (0.05)": p_value < 0.05
    })

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values("P-value").reset_index(drop=True)

print("Q3: Chi-square test results")
print("Total customers:", len(q3))
print("Customers with 3 or more products:", int((q3["target"] == "3_or_more").sum()))
print("Customers with fewer than 3 products:", int((q3["target"] == "fewer_than_3").sum()))
display(results_df.round(6))

if not results_df.empty:
    print("Most associated feature:", results_df.iloc[0]["Feature"])
    print("All tested variables significant at 0.05:",
          bool((results_df["P-value"] < 0.05).all()))
    print("Statistical association does not imply causation.")


Q3: Chi-square test results
Total customers: 931453
Customers with 3 or more products: 123021
Customers with fewer than 3 products: 808432


,Feature,Chi-square,Degrees of freedom,P-value,Significant (0.05)
0,Gender,3412.231367,1,0.0,True
1,Age group,61002.202414,2,0.0,True
2,Segment,124374.548799,2,0.0,True
3,Customer relation type,183808.591523,3,0.0,True
4,Employment status,1343.689088,4,0.0,True


Most associated feature: Gender
All tested variables significant at 0.05: True
Statistical association does not imply causation.


In [11]:
import zipfile
import os

zip_path = "/content/train_ver2.csv.zip"
csv_path = "/content/train_ver2.csv"

with zipfile.ZipFile(zip_path, "r") as z:
    csv_name = next(
        name for name in z.namelist()
        if name.endswith("train_ver2.csv")
    )
    with z.open(csv_name) as src, open(csv_path, "wb") as dst:
        import shutil
        shutil.copyfileobj(src, dst)

print("CSV:", os.path.exists(csv_path))

CSV: True
